# Linear Models of Returns

### Overview

This notebook shows a worked example of estimating the factor model given by:

$$\mathbf{r_t} = \boldsymbol{\alpha} + \mathbf{B} \mathbf{f_t} + \boldsymbol{\epsilon_t}$$

(equation (4.1) in (Paleologo (2025)), with $\mathbf{r_t}$ being the vector of asset excess returns, $\mathbf{B}$ the nxm loadings matrix, $\boldsymbol{\alpha}$ the n-dimensional alpha vector, $\mathbf{f_t}$ the vector of m factor returns and $\boldsymbol{\epsilon_t}$ the random vector of n idiosyncratic returns (Paleologo (2025)).

Our $\mathbf{B}$ comes from a factor model with 11 factors plus sector weights, calculated monthly for a subset of the S&P 500 stock index, with the following factors:

1. <b>Profitability</b> Operating profitability is (revenues - [Cost of Goods Sold] - [selling, general, and administrative expenses] - [interest expense])/[book equity] (Fama and French (2015)).

1. <b>Investment</b>
The change in Total Assets from the fiscal year ending in (t-2) to the fiscal year ending in (t-1), divided by (t-1) total assets.

1. <b>Value</b>
Defined as book value = Stockholders Equity / Market Capitalisation

1. <b>Size</b>
The market capitalisation. 

1. <b>Momentum</b>
Measuring price momentum, as the 12 month trailing return excluding the most recent month. 

1. <b>Beta</b> The beta from CAPM regressed against the broad S&P500 index over the previous 252 trading days.  

1. <b>Volatility</b> The residual volatility from the CAPM model calculated over the previous 252 trading days. 

1. <b>Growth</b> The growth in Total Revenue over the previous 12 months. 

1. <b>Leverage/Financial Health</b> Financial health is measured by Total Debt / Total Assets.

1. <b>Liquidity</b> Liquidity is measured as |daily return| / daily dollar volume

1. <b>Sector classification</b> Each sector is included in $B$ as a sparsely encoded column vector (1 if the stock is in that sector, 0 if not).

The factors were chosen to provide a broad coverage of fundamental and stylistic factor types, and the python module 'data_pipeline.py' has the calculation logic. 

### Roadmap

The sections that follow show how the factor data can be used to build a cross sectional $\mathbf{B}$ panel of factors for a given date, to allow estimation of $\mathbf{f_t}$ in the equation above. The notebook then goes on to show how this can be repeated over a number of dates, to allow the idiosyncratic covariance matrix $\boldsymbol{\Omega_\epsilon}$ and the factor covariance matrix $\boldsymbol{\Omega_f}$ to be estimated. Finally the fitted model is used to decompose a portfolio of returns. 

### Data
The stock population for this example is a subset of 427 of the stocks in the S&P 500, selected for survivability during the period 2010 - 2016, in the interests of simplifying the data ingestion. The list of tickers can be found in the factor_data folder. The survivorship bias this introduces is obviously considerable and the data would need to be fully adjusted for survivorship before these calculations could be moved into a production environment. Also, the data have not been thoroughly checked for corporate actions and events and again this would need to be done in a live system. The objective of this notebook is to build a working example of a factor model.  

To be conservative, a lag of 3 months was applied to any accounting data ingested to account for delays in filing. 

Fundamental and accounting data was accessed via Mathematica (which in turn curates a collection of institutional providers), with prices and volumes accessed from yahoo finance.

The raw inputs to the data_pipeline.py module have not been uploaded to github to honour licensing, but the derived factor values are available in the factor_data folder. 

### <u><b>Reference list</b></u>

Paleologo, G.A., 2025. The Elements of Quantitative Investing. John Wiley & Sons.

Fama, E.F. and French, K.R., 2015. A five-factor asset pricing model. Journal of financial economics, 116(1), pp.1-22.

In [13]:
import pandas as pd
import numpy as np

%load_ext autoreload
%autoreload 2
from factor_builder import FactorBuilder

We begin by creating a FactorBuilder class, a utility class that allows panel data to be easily calculated for a specified date. The factor data is available from each month end from January 2012 to March 2026.  

In [21]:
fb = FactorBuilder("./factor_data/factor_data.csv")
B = fb.build_cross_sectional_panel("2020-03-31")
B.head()

,beta,resid_vol,Value_BM,Profitability_OP,Investment_CMA,Momentum_12_1,Growth_Rev,Leverage_DA,Liquidity,MarketCap,...,DiversifiedFinancialServices,Energy,AutomobilesAndComponents,Banks,RealEstateManagementAndDevelopment,HouseholdAndPersonalProducts,MediaAndEntertainment,TelecommunicationServices,ConsumerStaplesDistributionAndRetail,ConsumerDurablesAndApparel
ticker,,,,,,,,,,,,,,,,,,,,,
A,1.657138,0.120905,-0.235678,-0.060982,-0.304060,-0.689516,-0.085793,-0.042118,-0.048422,-0.076860,...,0,0,0,0,0,0,0,0,0,0
AAL,0.977123,1.194700,-0.661608,0.062793,-0.694648,-2.012878,-0.084464,1.157012,-0.048365,-0.176393,...,0,0,0,0,0,0,0,0,0,0
AAP,-1.147466,-0.095340,-0.279463,-0.048735,0.807921,0.364713,-0.164638,-0.620875,-0.048419,-0.265222,...,0,0,0,0,0,0,0,0,0,0
AAPL,-0.879686,-0.169893,-0.264743,-0.053644,1.564030,0.775021,0.820482,-1.392509,-0.048440,4.571753,...,0,0,0,0,0,0,0,0,0,0
ABT,-1.464059,-0.458598,-0.063464,-0.071997,-0.030074,1.128254,-1.677892,0.185553,-0.048438,0.138870,...,0,0,0,0,0,0,0,0,0,0


We also need prices for the regression. These are resampled to month end dates to be consistent with the factor data:


In [ ]:
def forward_return_vector(date, daily_prices, tickers=None):
    # utility function to create the forward return vector for a given date, the vector of returns from t to t+1
    date = pd.to_datetime(date)
    next_month_end = date + pd.offsets.MonthEnd(1)

    start_prices = daily_prices.loc[date]
    end_prices = daily_prices.loc[:next_month_end].iloc[-1]

    r = end_prices / start_prices - 1

    if tickers is not None:
        r = r.reindex(tickers)
    return r